In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import os

UPSTREAM_COMMIT = "b76b93da8bd3096a99964a96ae29d46f197a0651"
PROJECT_ROOT = Path("/content/drive/MyDrive/CS2203_MuSc")
RUNS_ROOT = PROJECT_ROOT / "runs"
ACTIVE_RUN_FILE = PROJECT_ROOT / "active-e02-run.txt"

CATEGORIES = [
    "candle", "capsules", "cashew", "chewinggum", "fryum",
    "macaroni1", "macaroni2", "pcb1", "pcb2", "pcb3",
    "pcb4", "pipe_fryum",
]

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if ACTIVE_RUN_FILE.is_file():
    candidate_run_id = ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip()
    candidate_root = RUNS_ROOT / candidate_run_id
    if candidate_root.is_dir() and not (candidate_root / "E02_AGGREGATED.json").exists():
        RUN_ID = candidate_run_id
        print("Tiếp tục E02 Run ID đang dở:", RUN_ID)
    else:
        ACTIVE_RUN_FILE.unlink()
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E02_visa_all_clip_l14_336_baseline")
        ACTIVE_RUN_FILE.write_text(RUN_ID + "\n", encoding="utf-8")
        print("Tạo E02 Run ID mới:", RUN_ID)
else:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E02_visa_all_clip_l14_336_baseline")
    ACTIVE_RUN_FILE.write_text(RUN_ID + "\n", encoding="utf-8")
    print("Tạo E02 Run ID mới:", RUN_ID)

RUN_ROOT = RUNS_ROOT / RUN_ID
CATEGORY_ROOT = RUN_ROOT / "categories"
SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SESSION_ROOT = RUN_ROOT / "sessions" / SESSION_ID

CATEGORY_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_ROOT.mkdir(parents=True, exist_ok=False)

os.environ["MUSC_COMMIT"] = UPSTREAM_COMMIT
os.environ["MUSC_E02_RUN_ROOT"] = str(RUN_ROOT)

(RUN_ROOT / "run-id.txt").write_text(RUN_ID + "\n", encoding="utf-8")

print("Run root:", RUN_ROOT)
print("Session root:", SESSION_ROOT)

Mounted at /content/drive
Tiếp tục E02 Run ID đang dở: 20261012_171400_E02_visa_all_clip_l14_336_baseline
Run root: /content/drive/MyDrive/CS2203_MuSc/runs/20261012_171400_E02_visa_all_clip_l14_336_baseline
Session root: /content/drive/MyDrive/CS2203_MuSc/runs/20261012_171400_E02_visa_all_clip_l14_336_baseline/sessions/20261013_162854


In [ ]:
import json
import pandas as pd

METRIC_COLUMNS = [
    "auroc_px", "f1_px", "ap_px", "aupro",
    "auroc_sp", "f1_sp", "ap_sp",
]

def successful_attempt(category):
    category_dir = CATEGORY_ROOT / category
    success_files = sorted(category_dir.glob("attempt_*/SUCCESS.json"))
    return success_files[-1].parent if success_files else None

completed = [c for c in CATEGORIES if successful_attempt(c) is not None]
missing = [c for c in CATEGORIES if successful_attempt(c) is None]

print("Đã hoàn thành:", len(completed), completed)
print("Còn thiếu:", missing)

assert len(completed) == 12, f"Chưa đủ 12 category: {missing}"

Đã hoàn thành: 6 ['candle', 'capsules', 'cashew', 'chewinggum', 'fryum', 'macaroni1']
Còn thiếu: ['macaroni2', 'pcb1', 'pcb2', 'pcb3', 'pcb4', 'pipe_fryum']


AssertionError: Chưa đủ 12 category: ['macaroni2', 'pcb1', 'pcb2', 'pcb3', 'pcb4', 'pipe_fryum']

In [ ]:
import platform
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "huggingface_hub",
        "ftfy==6.1.3",
        "openpyxl==3.1.2",
        "timm==0.9.12",
    ],
    check=True,
)

import cv2
import ftfy
import openpyxl
import sklearn
import skimage
import timm
import torch
import torchvision

assert torch.cuda.is_available(), "GPU chưa được bật hoặc Colab chưa cấp GPU."

versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "opencv": cv2.__version__,
    "ftfy": ftfy.__version__,
    "openpyxl": openpyxl.__version__,
    "scikit_learn": sklearn.__version__,
    "scikit_image": skimage.__version__,
    "timm": timm.__version__,
    "upstream_commit": UPSTREAM_COMMIT,
}

environment_text = "\n".join(f"{key}={value}" for key, value in versions.items())
(SESSION_ROOT / "environment.txt").write_text(environment_text + "\n", encoding="utf-8")
print(environment_text)
subprocess.run(["nvidia-smi"], check=True)

python=3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
platform=Linux-6.6.122+-x86_64-with-glibc2.39
torch=2.11.0+cu130
torchvision=0.26.0+cu130
cuda_runtime=13.0
gpu=Tesla T4
opencv=4.14.0
ftfy=6.1.2
openpyxl=3.1.2
scikit_learn=1.6.1
scikit_image=0.25.2
timm=0.9.12
upstream_commit=b76b93da8bd3096a99964a96ae29d46f197a0651


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
from datetime import datetime
import hashlib
import shutil
import subprocess

VISA_URL = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"
DRIVE_DATASET_CACHE = PROJECT_ROOT / "cache" / "visa"
DRIVE_DATASET_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_ARCHIVE = DRIVE_DATASET_CACHE / "VisA_20220922.tar"
LOCAL_ARCHIVE = Path("/content/VisA_20220922.tar")
LOCAL_PARTIAL = Path("/content/VisA_20220922.tar.partial")

def valid_tar(path):
    candidate = Path(path)
    if not candidate.is_file() or candidate.stat().st_size < 1_000_000_000:
        return False, "File thiếu hoặc nhỏ bất thường."
    result = subprocess.run(
        ["tar", "-tf", str(candidate)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    return result.returncode == 0, result.stderr.strip()

if DRIVE_ARCHIVE.is_file():
    print("Đang sao chép VisA archive từ Drive xuống /content...")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
    cached_valid, cached_message = valid_tar(LOCAL_ARCHIVE)
    print("Drive cache valid:", cached_valid)
    if not cached_valid:
        invalid_name = DRIVE_ARCHIVE.with_name(
            DRIVE_ARCHIVE.name + ".invalid-" + datetime.now().strftime("%Y%m%d_%H%M%S")
        )
        DRIVE_ARCHIVE.replace(invalid_name)
        LOCAL_ARCHIVE.unlink(missing_ok=True)
        print("Đã giữ cache lỗi tại:", invalid_name)
        print("Lý do:", cached_message)

if not LOCAL_ARCHIVE.is_file():
    LOCAL_PARTIAL.unlink(missing_ok=True)
    print("Đang tải VisA từ Amazon S3 chính thức...")
    subprocess.run(
        [
            "curl", "-L", "--fail", "--retry", "5",
            "--output", str(LOCAL_PARTIAL), VISA_URL,
        ],
        check=True,
    )
    downloaded_valid, downloaded_message = valid_tar(LOCAL_PARTIAL)
    print("Downloaded archive valid:", downloaded_valid)
    print("Archive check:", downloaded_message or "OK")
    assert downloaded_valid, "VisA archive tải xuống không hợp lệ."
    LOCAL_PARTIAL.replace(LOCAL_ARCHIVE)

local_valid, local_message = valid_tar(LOCAL_ARCHIVE)
assert local_valid, f"Local TAR không hợp lệ: {local_message}"

if not DRIVE_ARCHIVE.is_file() or DRIVE_ARCHIVE.stat().st_size != LOCAL_ARCHIVE.stat().st_size:
    print("Đang lưu archive hợp lệ vào Drive cache...")
    shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

sha256 = hashlib.sha256()
with LOCAL_ARCHIVE.open("rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)
archive_sha256 = sha256.hexdigest()

extract_root = Path("/content/visa-extract")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

print("Đang giải nén VisA...")
subprocess.run(
    ["tar", "-xf", str(LOCAL_ARCHIVE), "-C", str(extract_root)],
    check=True,
)

split_files = list(extract_root.rglob("split_csv/1cls.csv"))
assert len(split_files) == 1, f"Không xác định được duy nhất VisA root: {split_files}"
DATA_ROOT = split_files[0].parent.parent

for category in CATEGORIES:
    required_paths = [
        DATA_ROOT / category / "Data" / "Images" / "Normal",
        DATA_ROOT / category / "Data" / "Images" / "Anomaly",
        DATA_ROOT / category / "Data" / "Masks" / "Anomaly",
        DATA_ROOT / category / "image_anno.csv",
    ]
    for required in required_paths:
        assert required.exists(), f"Thiếu dữ liệu: {required}"

DATA_SOURCE = VISA_URL
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=VisA_20220922.tar\n"
    f"sha256={archive_sha256}\n"
    f"resolved_root={DATA_ROOT}\n"
)
dataset_source_path = RUN_ROOT / "dataset-source.txt"
if dataset_source_path.exists():
    previous = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous[0] == f"source={DATA_SOURCE}", "Nguồn dataset thay đổi khi resume."
    assert previous[2] == f"sha256={archive_sha256}", "SHA-256 dataset thay đổi khi resume."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Archive SHA-256:", archive_sha256)
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 12 category VisA.")

Đang sao chép VisA archive từ Drive xuống /content...
Drive cache valid: True
Đang giải nén VisA...
Archive SHA-256: 2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362
Dataset root: /content/visa-extract
Đã xác nhận đủ 12 category VisA.


In [ ]:
import shutil
import subprocess

REPO_ROOT = Path("/content/MuSc")
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", "https://github.com/xrli-U/MuSc.git", str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", UPSTREAM_COMMIT],
    cwd=REPO_ROOT,
    check=True,
)

actual_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout.strip()
assert actual_commit == UPSTREAM_COMMIT

entrypoint = REPO_ROOT / "examples" / "musc_main.py"
local_package = REPO_ROOT / "datasets" / "__init__.py"
source = entrypoint.read_bytes()
old_line = b"sys.path.append(os.getcwd())"
new_line = b"sys.path.insert(0, os.getcwd())"

if old_line in source:
    entrypoint.write_bytes(source.replace(old_line, new_line, 1))
elif new_line not in source:
    raise RuntimeError("Không tìm thấy dòng sys.path mong đợi trong musc_main.py")

local_package.touch(exist_ok=True)

diagnostic = subprocess.run(
    [
        sys.executable,
        "-c",
        "import datasets; print(datasets.__file__); "
        "import datasets.visa; print('Đã import datasets.visa thành công')",
    ],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
print(diagnostic.stdout)

resolved_package = Path(diagnostic.stdout.splitlines()[0]).resolve()
assert REPO_ROOT.resolve() in resolved_package.parents

repo_data_root = REPO_ROOT / "data"
repo_data_root.mkdir(parents=True, exist_ok=True)
visa_link = repo_data_root / "visa"
if visa_link.is_symlink() or visa_link.is_file():
    visa_link.unlink()
elif visa_link.is_dir():
    shutil.rmtree(visa_link)
visa_link.symlink_to(DATA_ROOT, target_is_directory=True)

subprocess.run(
    [sys.executable, "datasets/visa_preprocess.py"],
    cwd=REPO_ROOT,
    check=True,
)

meta_path = DATA_ROOT / "meta.json"
assert meta_path.is_file(), "visa_preprocess.py chưa tạo meta.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))
assert set(meta) == {"train", "test"}

test_counts = {}
for category in CATEGORIES:
    assert category in meta["test"], f"meta.json thiếu category {category}"
    test_counts[category] = len(meta["test"][category])
    assert test_counts[category] > 0, f"Test split rỗng: {category}"
    for sample in meta["test"][category]:
        assert (DATA_ROOT / sample["img_path"]).is_file(), sample["img_path"]
        if sample["mask_path"]:
            assert (DATA_ROOT / sample["mask_path"]).is_file(), sample["mask_path"]

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
git_diff = subprocess.run(
    ["git", "diff", "--", "examples/musc_main.py"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout

(SESSION_ROOT / "compatibility-fix.txt").write_text(
    "Reason: avoid collision with the Hugging Face datasets package.\n"
    "Effect on method: none; import resolution only.\n\n"
    + git_status + "\n" + git_diff,
    encoding="utf-8",
)
(RUN_ROOT / "upstream-commit.txt").write_text(actual_commit + "\n", encoding="utf-8")
(RUN_ROOT / "visa-test-counts.json").write_text(
    json.dumps(test_counts, indent=2) + "\n",
    encoding="utf-8",
)

print("VisA test counts:", test_counts)
print("Tổng số test image:", sum(test_counts.values()))
print("Đã tạo và kiểm tra meta.json.")

/content/MuSc/datasets/__init__.py
Đã import datasets.visa thành công

VisA test counts: {'candle': 200, 'capsules': 160, 'cashew': 150, 'chewinggum': 150, 'fryum': 150, 'macaroni1': 200, 'macaroni2': 200, 'pcb1': 200, 'pcb2': 200, 'pcb3': 201, 'pcb4': 201, 'pipe_fryum': 150}
Tổng số test image: 2162
Đã tạo và kiểm tra meta.json.


In [ ]:
import subprocess

musc_path = REPO_ROOT / "models" / "musc.py"
source = musc_path.read_bytes()
line_ending = b"\r\n" if b"\r\n" in source else b"\n"

def encode_block(text):
    return text.encode("utf-8").replace(b"\n", line_ending)

def replace_once(data, old_text, new_text, label):
    old = encode_block(old_text)
    new = encode_block(new_text)
    if new in data:
        print(f"SKIP {label}: đã áp dụng")
        return data
    count = data.count(old)
    assert count == 1, f"Không tìm thấy duy nhất marker {label}: count={count}"
    print(f"APPLY {label}")
    return data.replace(old, new, 1)

source = replace_once(source, "import cv2\n", "import cv2\nimport gc\n", "import gc")
source = replace_once(
    source,
    "        anomaly_maps = anomaly_maps.cpu().numpy()\n"
    "        torch.cuda.empty_cache()\n",
    "        anomaly_maps = anomaly_maps.cpu().numpy()\n"
    "        # Release feature tensors before allocating flattened metric arrays.\n"
    "        del patch_tokens_list, Z_layers, Z, features, patch_tokens\n"
    "        del anomaly_maps_l, anomaly_maps_msm, anomaly_maps_iter\n"
    "        del image_info, image, input_image, test_dataloader, test_dataset, LNAMD_r\n"
    "        gc.collect()\n"
    "        torch.cuda.empty_cache()\n",
    "release expired feature tensors",
)
source = replace_once(
    source,
    "        ac_score = np.array(anomaly_maps).reshape(B, -1).max(-1)\n",
    "        ac_score = anomaly_maps.reshape(B, -1).max(-1)\n",
    "remove anomaly-map copy for image score",
)
source = replace_once(
    source,
    "        pr_px = np.array(anomaly_maps)\n",
    "        pr_px = anomaly_maps\n",
    "remove anomaly-map copy for pixel metrics",
)
musc_path.write_bytes(source)

memory_diff = subprocess.run(
    ["git", "diff", "--", "models/musc.py"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
assert "gc.collect()" in memory_diff
assert "ac_score = anomaly_maps.reshape" in memory_diff
assert "pr_px = anomaly_maps" in memory_diff

(SESSION_ROOT / "cpu-memory-patch.diff").write_text(memory_diff, encoding="utf-8")
print(memory_diff)

APPLY import gc
APPLY release expired feature tensors
APPLY remove anomaly-map copy for image score
APPLY remove anomaly-map copy for pixel metrics
diff --git a/models/musc.py b/models/musc.py
index 92e91d9..a94213f 100644
--- a/models/musc.py
+++ b/models/musc.py
@@ -23,6 +23,7 @@ from tqdm import tqdm
 import pickle
 import time
 import cv2
+import gc
 
 import warnings
 warnings.filterwarnings("ignore")
@@ -234,10 +235,15 @@ class MuSc():
         print('MuSc: {}ms per image'.format((end_time_all-start_time_all)*1000/dataset_num))
 
         anomaly_maps = anomaly_maps.cpu().numpy()
+        # Release feature tensors before allocating flattened metric arrays.
+        del patch_tokens_list, Z_layers, Z, features, patch_tokens
+        del anomaly_maps_l, anomaly_maps_msm, anomaly_maps_iter
+        del image_info, image, input_image, test_dataloader, test_dataset, LNAMD_r
+        gc.collect()
         torch.cuda.empty_cache()
 
         B = anomaly_maps.shape[0]   # the number of u

In [ ]:
import shutil

LOCAL_CLIP_CACHE = Path.home() / ".cache" / "clip"
DRIVE_CLIP_CACHE = PROJECT_ROOT / "cache" / "clip"
LOCAL_CLIP_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CLIP_CACHE.mkdir(parents=True, exist_ok=True)

restored_files = []
for source_file in DRIVE_CLIP_CACHE.glob("*"):
    if source_file.is_file():
        target_file = LOCAL_CLIP_CACHE / source_file.name
        if not target_file.exists() or target_file.stat().st_size != source_file.stat().st_size:
            shutil.copy2(source_file, target_file)
        restored_files.append(target_file.name)

if restored_files:
    print("Đã khôi phục CLIP cache:", restored_files)
else:
    print("Chưa có CLIP cache trong Drive. Category đầu tiên sẽ tải weights khoảng 934 MB.")

Đã khôi phục CLIP cache: ['ViT-L-14-336px.pt']


In [ ]:
# lưu cấu hình baseline
BASELINE_CONFIG = {
    "dataset_name": "visa",
    "backbone_name": "ViT-L-14-336",
    "pretrained": "openai",
    "feature_layers": [5, 11, 17, 23],
    "img_resize": 518,
    "divide_num": 1,
    "r_list": [1, 3, 5],
    "batch_size": 4,
    "vis": False,
    "vis_type": "whole_norm",
    "save_excel": True,
    "seed_in_upstream_entrypoint": 42,
}

config_path = RUN_ROOT / "baseline-config.json"
config_text = json.dumps(BASELINE_CONFIG, indent=2, sort_keys=True) + "\n"
if config_path.exists():
    assert config_path.read_text(encoding="utf-8") == config_text, "Baseline config đã thay đổi."
else:
    config_path.write_text(config_text, encoding="utf-8")

print(config_text)

{
  "backbone_name": "ViT-L-14-336",
  "batch_size": 4,
  "dataset_name": "visa",
  "divide_num": 1,
  "feature_layers": [
    5,
    11,
    17,
    23
  ],
  "img_resize": 518,
  "pretrained": "openai",
  "r_list": [
    1,
    3,
    5
  ],
  "save_excel": true,
  "seed_in_upstream_entrypoint": 42,
  "vis": false,
  "vis_type": "whole_norm"
}



In [ ]:
from datetime import datetime, timezone
import json
import math
import pandas as pd
import shlex
import shutil
import subprocess
import time

METRIC_COLUMNS = [
    "auroc_px", "f1_px", "ap_px", "aupro",
    "auroc_sp", "f1_sp", "ap_sp",
]

def sync_clip_cache_to_drive():
    DRIVE_CLIP_CACHE.mkdir(parents=True, exist_ok=True)
    for source_file in LOCAL_CLIP_CACHE.glob("*"):
        if source_file.is_file():
            target_file = DRIVE_CLIP_CACHE / source_file.name
            if not target_file.exists() or target_file.stat().st_size != source_file.stat().st_size:
                shutil.copy2(source_file, target_file)

def successful_attempt(category):
    category_dir = CATEGORY_ROOT / category
    success_files = sorted(category_dir.glob("attempt_*/SUCCESS.json"))
    return success_files[-1].parent if success_files else None

def run_category(category):
    assert category in CATEGORIES, f"Category không hợp lệ: {category}"

    previous_success = successful_attempt(category)
    if previous_success is not None:
        metrics = json.loads((previous_success / "metrics.json").read_text(encoding="utf-8"))
        print(f"SKIP {category}: đã thành công tại {previous_success}")
        return metrics

    category_dir = CATEGORY_ROOT / category
    category_dir.mkdir(parents=True, exist_ok=True)
    attempt_number = len(list(category_dir.glob("attempt_*"))) + 1
    attempt_root = category_dir / f"attempt_{attempt_number:02d}"
    output_root = attempt_root / "output"
    attempt_root.mkdir(parents=True, exist_ok=False)

    command = [
        sys.executable,
        "examples/musc_main.py",
        "--device", "0",
        "--data_path", str(DATA_ROOT),
        "--dataset_name", "visa",
        "--class_name", category,
        "--backbone_name", "ViT-L-14-336",
        "--pretrained", "openai",
        "--feature_layers", "5", "11", "17", "23",
        "--img_resize", "518",
        "--divide_num", "1",
        "--r_list", "1", "3", "5",
        "--batch_size", "4",
        "--output_dir", str(output_root),
        "--vis", "False",
        "--vis_type", "whole_norm",
        "--save_excel", "True",
    ]

    (attempt_root / "command.sh").write_text(
        "#!/usr/bin/env bash\nset -euo pipefail\n" + shlex.join(command) + "\n",
        encoding="utf-8",
    )

    started_at = datetime.now(timezone.utc)
    started = time.time()
    log_path = attempt_root / "console.log"

    print(f"START {category}, attempt {attempt_number:02d}")
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()

    elapsed_seconds = time.time() - started
    sync_clip_cache_to_drive()

    status = {
        "category": category,
        "attempt": attempt_number,
        "started_at_utc": started_at.isoformat(),
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": elapsed_seconds,
        "return_code": return_code,
        "gpu": torch.cuda.get_device_name(0),
    }

    if return_code != 0:
        (attempt_root / "FAILED.json").write_text(
            json.dumps(status, indent=2) + "\n",
            encoding="utf-8",
        )
        raise RuntimeError(f"{category} thất bại. Xem log: {log_path}")

    result_file = (
        output_root / "visa" / "ViT-L-14-336" /
        "imagesize518" / "results.xlsx"
    )
    assert result_file.is_file(), f"Thiếu file kết quả: {result_file}"

    frame = pd.read_excel(result_file)
    frame = frame.rename(columns={frame.columns[0]: "category"})
    row = frame.iloc[0]
    assert row["category"] == category

    metrics = {"category": category}
    metrics.update({column: float(row[column]) for column in METRIC_COLUMNS})
    for column in METRIC_COLUMNS:
        assert math.isfinite(metrics[column]) and 0 <= metrics[column] <= 100
    metrics["elapsed_seconds"] = elapsed_seconds
    metrics["attempt_root"] = str(attempt_root)

    (attempt_root / "metrics.json").write_text(
        json.dumps(metrics, indent=2) + "\n",
        encoding="utf-8",
    )
    (attempt_root / "SUCCESS.json").write_text(
        json.dumps(status, indent=2) + "\n",
        encoding="utf-8",
    )

    print(f"DONE {category}: {elapsed_seconds / 60:.2f} phút")
    print({column: round(metrics[column], 4) for column in METRIC_COLUMNS})
    return metrics

In [ ]:
# cel 08
CANDLE_REFERENCE = {
    "auroc_sp": 96.55,
    "f1_sp": 91.26,
    "ap_sp": 96.45,
    "auroc_px": 99.36,
    "f1_px": 39.56,
    "ap_px": 28.36,
    "aupro": 97.62,
}

candle_metrics = run_category("candle")
candle_deltas = {
    metric: candle_metrics[metric] - CANDLE_REFERENCE[metric]
    for metric in METRIC_COLUMNS
}

print("Chênh lệch candle với repository:")
for metric, delta in candle_deltas.items():
    print(f"  {metric}: {delta:+.6f}")

(RUN_ROOT / "candle-smoke-check.json").write_text(
    json.dumps(
        {
            "category": "candle",
            "deltas_from_repository": candle_deltas,
            "status": "completed",
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

START candle, attempt 01
xFormers not available
xFormers not available
{'datasets': {'dataset_name': 'visa', 'data_path': '/content/visa-extract', 'class_name': 'candle', 'img_resize': 518, 'divide_num': 1}, 'models': {'backbone_name': 'ViT-L-14-336', 'pretrained': 'openai', 'batch_size': 4, 'feature_layers': [5, 11, 17, 23], 'r_list': [1, 3, 5]}, 'device': '0', 'testing': {'output_dir': '/content/drive/MyDrive/CS2203_MuSc/runs/20261012_171400_E02_visa_all_clip_l14_336_baseline/categories/candle/attempt_01/output', 'vis': False, 'vis_type': 'whole_norm', 'save_excel': True}}
candle

100%|██████████| 50/50 [00:43<00:00,  1.15it/s]
extract time: 216.92026734352112ms per image
aggregation degree: 1
LNAMD-1: 32.60369420051575ms per image
layer-0 mutual scoring...

100%|██████████| 200/200 [00:44<00:00,  4.54it/s]
layer-1 mutual scoring...

100%|██████████| 200/200 [00:49<00:00,  4.07it/s]
layer-2 mutual scoring...

100%|██████████| 200/200 [00:48<00:00,  4.13it/s]
layer-3 mutual scoring...

337

In [ ]:
# cell 8R
from collections import deque

progress_rows = []
for category in CATEGORIES:
    category_dir = CATEGORY_ROOT / category
    attempts = sorted(category_dir.glob("attempt_*"))
    success = successful_attempt(category)

    if success is not None:
        status = "SUCCESS"
        latest = success
    elif attempts:
        latest = attempts[-1]
        status = "FAILED" if (latest / "FAILED.json").is_file() else "INTERRUPTED"
    else:
        latest = None
        status = "NOT_STARTED"

    progress_rows.append(
        {
            "category": category,
            "status": status,
            "attempts": len(attempts),
            "latest_attempt": str(latest) if latest else "",
        }
    )

progress = pd.DataFrame(progress_rows)
display(progress)

interrupted = progress.loc[progress["status"] == "INTERRUPTED", "category"].tolist()
remaining = progress.loc[progress["status"] != "SUCCESS", "category"].tolist()
print("Category bị gián đoạn:", interrupted)
print("Category còn lại:", remaining)

for category in interrupted:
    latest_attempt = Path(
        progress.loc[progress["category"] == category, "latest_attempt"].iloc[0]
    )
    log_path = latest_attempt / "console.log"
    if log_path.is_file():
        with log_path.open("r", encoding="utf-8", errors="replace") as stream:
            tail = list(deque(stream, maxlen=12))
        print(f"\n===== Cuối log của {category} =====")
        print("".join(tail))

,category,status,attempts,latest_attempt
0,candle,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
1,capsules,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
2,cashew,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
3,chewinggum,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
4,fryum,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
5,macaroni1,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
6,macaroni2,INTERRUPTED,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
7,pcb1,NOT_STARTED,0,
8,pcb2,NOT_STARTED,0,
9,pcb3,NOT_STARTED,0,


Category bị gián đoạn: ['macaroni2']
Category còn lại: ['macaroni2', 'pcb1', 'pcb2', 'pcb3', 'pcb4', 'pipe_fryum']

===== Cuối log của macaroni2 =====
 72%|███████▏  | 143/200 [00:30<00:17,  3.18it/s]



In [ ]:
# cell 9 - chạy 12 category
all_metrics = {}

for index, category in enumerate(CATEGORIES, start=1):
    print(f"\n===== [{index:02d}/{len(CATEGORIES)}] {category} =====")
    all_metrics[category] = run_category(category)

print("Đã hoàn thành đủ 12 category VisA.")

Streaming output truncated to the last 5000 lines.
100%|██████████| 200/200 [00:53<00:00,  3.77it/s]
layer-3 mutual scoring...

100%|██████████| 200/200 [00:52<00:00,  3.82it/s]
MSM: 1396.7479479312897ms per image
aggregation degree: 5
LNAMD-5: 55.32298922538757ms per image
layer-0 mutual scoring...

100%|██████████| 200/200 [00:53<00:00,  3.74it/s]
layer-1 mutual scoring...

100%|██████████| 200/200 [00:53<00:00,  3.75it/s]
layer-2 mutual scoring...

100%|██████████| 200/200 [00:52<00:00,  3.81it/s]
layer-3 mutual scoring...

100%|██████████| 200/200 [00:52<00:00,  3.81it/s]
MSM: 1395.0475251674652ms per image
MuSc: 4576.915602684021ms per image
computing metrics...
pcb2
image-level, auroc:92.00999999999999, f1:86.15384615384616, ap:93.57310299686175
pixel-level, auroc:97.38969336990932, f1:34.380835955991365, ap:21.854665537864868, aupro:86.04891012115137
pcb2
image-level, auroc:92.00999999999999, f1:86.15384615384616, ap:93.57310299686175
pixel-level, auroc:97.38969336990932, f1:34.

In [ ]:
# tổng hợp kết quả
records = []
missing_categories = []

for category in CATEGORIES:
    attempt_root = successful_attempt(category)
    if attempt_root is None:
        missing_categories.append(category)
        continue
    records.append(json.loads((attempt_root / "metrics.json").read_text(encoding="utf-8")))

assert not missing_categories, f"Chưa hoàn thành: {missing_categories}"
assert len(records) == 12

reproduced = pd.DataFrame(records)
reproduced["category"] = pd.Categorical(
    reproduced["category"], categories=CATEGORIES, ordered=True
)
reproduced = reproduced.sort_values("category").reset_index(drop=True)

mean_row = {"category": "mean"}
for column in METRIC_COLUMNS:
    mean_row[column] = reproduced[column].mean()
mean_row["elapsed_seconds"] = reproduced["elapsed_seconds"].sum()
mean_row["attempt_root"] = ""

reproduced_with_mean = pd.concat(
    [reproduced, pd.DataFrame([mean_row])], ignore_index=True
)

reproduced_with_mean.to_csv(RUN_ROOT / "visa_reproduced.csv", index=False)
reproduced_with_mean.to_excel(RUN_ROOT / "visa_reproduced.xlsx", index=False)

display(reproduced_with_mean)
print("Tổng thời gian category subprocess:", reproduced["elapsed_seconds"].sum() / 60, "phút")

,category,auroc_px,f1_px,ap_px,aupro,auroc_sp,f1_sp,ap_sp,elapsed_seconds,attempt_root
0,candle,99.356979,39.575749,28.369788,97.607395,96.530000,90.909091,96.358649,1118.055894,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
1,capsules,98.714604,50.866861,43.922851,88.198029,88.366667,86.138614,93.677121,781.079109,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
2,cashew,99.332253,74.876593,77.618268,94.283970,98.280000,95.384615,99.191532,698.055457,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
3,chewinggum,99.538583,61.335368,61.215592,88.347657,98.420000,95.959596,99.301213,686.532535,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
4,fryum,97.930904,58.130141,50.429013,94.376207,98.580000,97.435897,99.403763,693.304680,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
5,macaroni1,99.507609,21.905208,15.257719,96.370933,89.500000,82.758621,88.424538,1099.916279,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
6,macaroni2,97.143885,11.071199,3.911966,88.842514,67.990000,70.212766,67.360317,1172.115606,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
7,pcb1,99.497902,80.492727,88.359356,92.767963,89.090000,84.210526,89.712354,1182.508590,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
8,pcb2,97.389693,34.380836,21.854666,86.048910,92.010000,86.153846,93.573103,1178.224933,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
9,pcb3,98.052778,40.229923,41.030378,92.324566,92.970297,85.981308,93.279338,1177.808748,/content/drive/MyDrive/CS2203_MuSc/runs/202610...


Tổng thời gian category subprocess: 194.93963282903036 phút


In [ ]:
# so sánh với bảng visa của repo
reference_records = [
    {"category": "candle", "auroc_sp": 96.55, "f1_sp": 91.26, "ap_sp": 96.45, "auroc_px": 99.36, "f1_px": 39.56, "ap_px": 28.36, "aupro": 97.62},
    {"category": "capsules", "auroc_sp": 88.62, "f1_sp": 86.43, "ap_sp": 93.77, "auroc_px": 98.71, "f1_px": 50.85, "ap_px": 43.90, "aupro": 88.20},
    {"category": "cashew", "auroc_sp": 98.54, "f1_sp": 95.57, "ap_sp": 99.30, "auroc_px": 99.33, "f1_px": 74.88, "ap_px": 77.63, "aupro": 94.30},
    {"category": "chewinggum", "auroc_sp": 98.42, "f1_sp": 96.45, "ap_sp": 99.30, "auroc_px": 99.54, "f1_px": 61.33, "ap_px": 61.21, "aupro": 88.39},
    {"category": "fryum", "auroc_sp": 98.64, "f1_sp": 97.44, "ap_sp": 99.43, "auroc_px": 99.43, "f1_px": 58.13, "ap_px": 50.43, "aupro": 94.38},
    {"category": "macaroni1", "auroc_sp": 89.33, "f1_sp": 82.76, "ap_sp": 88.64, "auroc_px": 99.51, "f1_px": 21.90, "ap_px": 15.25, "aupro": 96.37},
    {"category": "macaroni2", "auroc_sp": 68.03, "f1_sp": 69.96, "ap_sp": 67.37, "auroc_px": 97.14, "f1_px": 11.06, "ap_px": 3.91, "aupro": 88.84},
    {"category": "pcb1", "auroc_sp": 89.28, "f1_sp": 84.36, "ap_sp": 89.89, "auroc_px": 99.50, "f1_px": 80.49, "ap_px": 88.36, "aupro": 92.76},
    {"category": "pcb2", "auroc_sp": 93.20, "f1_sp": 88.66, "ap_sp": 94.46, "auroc_px": 97.39, "f1_px": 34.38, "ap_px": 21.86, "aupro": 86.06},
    {"category": "pcb3", "auroc_sp": 93.52, "f1_sp": 86.92, "ap_sp": 93.48, "auroc_px": 98.05, "f1_px": 40.23, "ap_px": 41.03, "aupro": 92.32},
    {"category": "pcb4", "auroc_sp": 98.43, "f1_sp": 92.89, "ap_sp": 98.47, "auroc_px": 98.70, "f1_px": 46.38, "ap_px": 44.72, "aupro": 92.66},
    {"category": "pipe_fryum", "auroc_sp": 98.34, "f1_sp": 96.04, "ap_sp": 99.16, "auroc_px": 99.40, "f1_px": 67.56, "ap_px": 67.90, "aupro": 97.32},
]

repository_mean = {
    "category": "mean",
    "auroc_sp": 92.57,
    "f1_sp": 89.06,
    "ap_sp": 93.31,
    "auroc_px": 98.71,
    "f1_px": 48.90,
    "ap_px": 45.38,
    "aupro": 92.43,
}

reference = pd.DataFrame(reference_records + [repository_mean])
comparison = reproduced_with_mean[["category", *METRIC_COLUMNS]].merge(
    reference[["category", *METRIC_COLUMNS]],
    on="category",
    suffixes=("_reproduced", "_reference"),
    validate="one_to_one",
)

for metric in METRIC_COLUMNS:
    comparison[f"{metric}_delta"] = (
        comparison[f"{metric}_reproduced"] - comparison[f"{metric}_reference"]
    )

delta_columns = [f"{metric}_delta" for metric in METRIC_COLUMNS]
category_rows = comparison[comparison["category"] != "mean"]
max_absolute_delta = category_rows[delta_columns].abs().to_numpy().max()

alerts = []
for _, row in category_rows.iterrows():
    for metric in METRIC_COLUMNS:
        delta = float(row[f"{metric}_delta"])
        if abs(delta) > 0.10:
            alerts.append(
                {
                    "category": row["category"],
                    "metric": metric,
                    "reproduced": row[f"{metric}_reproduced"],
                    "reference": row[f"{metric}_reference"],
                    "delta": delta,
                }
            )

comparison.to_csv(RUN_ROOT / "visa_reference_comparison.csv", index=False)
comparison.to_excel(RUN_ROOT / "visa_reference_comparison.xlsx", index=False)
pd.DataFrame(alerts).to_csv(RUN_ROOT / "comparison-alerts.csv", index=False)

display(comparison[comparison["category"] == "mean"])
print(f"Sai khác tuyệt đối lớn nhất ở category rows: {max_absolute_delta:.6f}")
print(f"Số cảnh báo vượt 0.10 điểm phần trăm: {len(alerts)}")

summary_lines = [
    "# E02 VisA Baseline Summary",
    "",
    f"- Run ID: `{RUN_ID}`",
    f"- Completed categories: {len(records)}/12",
    f"- Maximum absolute category-metric delta: {max_absolute_delta:.6f} percentage points",
    f"- Deltas above 0.10 percentage points: {len(alerts)}",
    "",
    "## Reproduced Mean",
    "",
]

mean_record = reproduced_with_mean[reproduced_with_mean["category"] == "mean"].iloc[0]
for metric in METRIC_COLUMNS:
    summary_lines.append(f"- {metric}: {mean_record[metric]:.6f}")

(RUN_ROOT / "E02_SUMMARY.md").write_text(
    "\n".join(summary_lines) + "\n", encoding="utf-8"
)

completion = {
    "run_id": RUN_ID,
    "aggregated_at_utc": datetime.now(timezone.utc).isoformat(),
    "completed_categories": len(records),
    "max_absolute_category_metric_delta": float(max_absolute_delta),
    "alerts_over_0_10_percentage_points": len(alerts),
}
(RUN_ROOT / "E02_AGGREGATED.json").write_text(
    json.dumps(completion, indent=2) + "\n", encoding="utf-8"
)

if ACTIVE_RUN_FILE.is_file() and ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip() == RUN_ID:
    ACTIVE_RUN_FILE.unlink()

,category,auroc_px_reproduced,f1_px_reproduced,ap_px_reproduced,aupro_reproduced,auroc_sp_reproduced,f1_sp_reproduced,ap_sp_reproduced,auroc_px_reference,f1_px_reference,...,auroc_sp_reference,f1_sp_reference,ap_sp_reference,auroc_px_delta,f1_px_delta,ap_px_delta,aupro_delta,auroc_sp_delta,f1_sp_delta,ap_sp_delta
12,mean,98.713645,48.900484,45.383142,92.428779,92.400077,88.797846,93.177198,98.71,48.9,...,92.57,89.06,93.31,0.003645,0.000484,0.003142,-0.001221,-0.169923,-0.262154,-0.132802


Sai khác tuyệt đối lớn nhất ở category rows: 2.506154
Số cảnh báo vượt 0.10 điểm phần trăm: 25


In [ ]:
# đóng gói kết quả
import shutil

EXPORT_ROOT = PROJECT_ROOT / "exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
archive_path = shutil.make_archive(
    str(EXPORT_ROOT / RUN_ID),
    "zip",
    root_dir=RUN_ROOT,
)

print("Đã tạo ZIP:", archive_path)
print("Dung lượng ZIP:", Path(archive_path).stat().st_size / 2**20, "MiB")

Đã tạo ZIP: /content/drive/MyDrive/CS2203_MuSc/exports/20261012_171400_E02_visa_all_clip_l14_336_baseline.zip
Dung lượng ZIP: 0.19686412811279297 MiB
